In [1]:
import motile
import plotly.io as pio
pio.renderers.default = "sphinx_gallery"
import networkx as nx

cells = [
        {"id": 0, "t": 0, "x": 1, "score": 0.8},
        {"id": 1, "t": 0, "x": 25, "score": 0.1},
        {"id": 2, "t": 1, "x": 0, "score": 0.3},
        {"id": 3, "t": 1, "x": 26, "score": 0.4},
        {"id": 4, "t": 2, "x": 2, "score": 0.6},
        {"id": 5, "t": 2, "x": 24, "score": 0.3},
        {"id": 6, "t": 2, "x": 35, "score": 0.7}
]

edges = [
    {"source": 0, "target": 2, "score": 0.9},
    {"source": 1, "target": 3, "score": 0.9},
    {"source": 0, "target": 3, "score": 0.5},
    {"source": 1, "target": 2, "score": 0.5},
    {"source": 2, "target": 4, "score": 0.7},
    {"source": 3, "target": 5, "score": 0.7},
    {"source": 2, "target": 5, "score": 0.3},
    {"source": 3, "target": 4, "score": 0.3},
    {"source": 3, "target": 6, "score": 0.8}
]

graph = nx.DiGraph()
graph.add_nodes_from([
    (cell["id"], cell)
    for cell in cells
])
graph.add_edges_from([
    (edge["source"], edge["target"], edge)
    for edge in edges
])

graph = motile.TrackGraph(graph)

In [2]:
from motile_toolbox.visualization import draw_track_graph

draw_track_graph(graph, alpha_attribute="score", label_attribute="score")

In [3]:
from motile.constraints import MaxParents, MaxChildren
from motile.costs import NodeSelection, EdgeSelection, Appear

solver = motile.Solver(graph)

solver.add_constraint(MaxParents(1))
solver.add_constraint(MaxChildren(1))

solver.add_cost(NodeSelection(weight=1, attribute="score"))
solver.add_cost(EdgeSelection(weight=-1, attribute="score", constant=0.5))
solver.add_cost(Appear(constant=1))

In [4]:
solver.weights

('NodeSelection', 'weight') = 1
('NodeSelection', 'constant') = 0.0
('EdgeSelection', 'weight') = -1
('EdgeSelection', 'constant') = 0.5
('Appear', 'weight') = 1
('Appear', 'constant') = 1

In [5]:
from motile.variables import NodeSelected, EdgeSelected
from motile_toolbox.visualization import draw_solution

solver.solve()

Solution(variable_values=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], objective_value=0.0, status=<SolverStatus.OPTIMAL: 'optimal'>, time=0.0, native_status='optimal')

In [6]:
print(solver.get_variables(NodeSelected))
print(solver.get_variables(EdgeSelected))

draw_solution(graph, solver, label_attribute="score")

NodeSelected(0): cost=0.800000011920929 value=0.0
NodeSelected(1): cost=0.10000000149011612 value=0.0
NodeSelected(2): cost=0.30000001192092896 value=0.0
NodeSelected(3): cost=0.4000000059604645 value=0.0
NodeSelected(4): cost=0.6000000238418579 value=0.0
NodeSelected(5): cost=0.30000001192092896 value=0.0
NodeSelected(6): cost=0.699999988079071 value=0.0
EdgeSelected((0, 2)): cost=-0.3999999761581421 value=0.0
EdgeSelected((0, 3)): cost=0.0 value=0.0
EdgeSelected((1, 3)): cost=-0.3999999761581421 value=0.0
EdgeSelected((1, 2)): cost=0.0 value=0.0
EdgeSelected((2, 4)): cost=-0.19999998807907104 value=0.0
EdgeSelected((2, 5)): cost=0.19999998807907104 value=0.0
EdgeSelected((3, 5)): cost=-0.19999998807907104 value=0.0
EdgeSelected((3, 4)): cost=0.19999998807907104 value=0.0
EdgeSelected((3, 6)): cost=-0.30000001192092896 value=0.0


In [7]:
solver.weights[("EdgeSelection", "weight")] = -2

solution = solver.solve()

In [8]:
print(solver.weights)
print(solver.get_variables(NodeSelected))
print(solver.get_variables(EdgeSelected))

draw_solution(graph, solver, label_attribute="score")

('NodeSelection', 'weight') = 1
('NodeSelection', 'constant') = 0.0
('EdgeSelection', 'weight') = -2
('EdgeSelection', 'constant') = 0.5
('Appear', 'weight') = 1
('Appear', 'constant') = 1

NodeSelected(0): cost=0.800000011920929 value=1.0
NodeSelected(1): cost=0.10000000149011612 value=1.0
NodeSelected(2): cost=0.30000001192092896 value=1.0
NodeSelected(3): cost=0.4000000059604645 value=1.0
NodeSelected(4): cost=0.6000000238418579 value=1.0
NodeSelected(5): cost=0.30000001192092896 value=1.0
NodeSelected(6): cost=0.699999988079071 value=0.0
EdgeSelected((0, 2)): cost=-1.2999999523162842 value=1.0
EdgeSelected((0, 3)): cost=-0.5 value=0.0
EdgeSelected((1, 3)): cost=-1.2999999523162842 value=1.0
EdgeSelected((1, 2)): cost=-0.5 value=0.0
EdgeSelected((2, 4)): cost=-0.8999999761581421 value=1.0
EdgeSelected((2, 5)): cost=-0.10000002384185791 value=0.0
EdgeSelected((3, 5)): cost=-0.8999999761581421 value=1.0
EdgeSelected((3, 4)): cost=-0.10000002384185791 value=0.0
EdgeSelected((3, 6)): co

In [9]:
graph.nodes[0]["gt"] = True
graph.nodes[2]["gt"] = True
graph.nodes[4]["gt"] = True
graph.nodes[5]["gt"] = False

graph.edges[(0, 2)]["gt"] = True
graph.edges[(2, 4)]["gt"] = True
graph.edges[(2, 5)]["gt"] = False

In [10]:
node_colors = [
  (0, 0, 0) if "gt" not in node else ((0, 128, 0) if node["gt"] else (255, 140, 0))
  for node in graph.nodes.values()
]
edge_colors = [
  (0, 0, 0) if "gt" not in edge else ((0, 128, 0) if edge["gt"] else (255, 140, 0))
  for edge in graph.edges.values()
]

draw_track_graph(
  graph,
  alpha_func=(
    lambda x: "gt" in graph.nodes[x],
    lambda x: "gt" in graph.edges[x]),
  node_color=node_colors,
  edge_color=edge_colors)

In [11]:
# this suppresses logging output from structsvm that can fail the docs build
import logging
logging.getLogger("structsvm.bundle_method").setLevel(logging.CRITICAL)

In [12]:
solver.fit_weights(gt_attribute="gt", regularizer_weight=0.01)
optimal_weights = solver.weights

presolving:
   (0.0s) symmetry computation started: requiring (bin +, int +, cont +), (fixed: bin -, int -, cont -)
   (0.0s) no symmetry present (symcode time: 0.00)
presolving (1 rounds: 1 fast, 1 medium, 1 exhaustive):
 0 deleted vars, 0 deleted constraints, 0 added constraints, 0 tightened bounds, 0 added holes, 0 changed sides, 0 changed coefficients
 0 implications, 0 cliques, 0 implied integral variables (0 bin, 0 int, 0 cont)
presolved problem has 8 variables (0 bin, 0 int, 8 cont) and 2 constraints
      1 constraints of type <linear>
      1 constraints of type <nonlinear>
Presolving Time: 0.00

 time | node  | left  |LP iter|LP it/n|mem/heur|mdpt |vars |cons |rows |cuts |sepa|confs|strbr|  dualbound   | primalbound  |  gap   | compl. 
t 0.0s|     1 |     0 |     0 |     - |  trysol|   0 |   8 |   2 |   0 |   0 |  0 |   0 |   0 |      --      | 2.500000e+08 |    Inf | unknown
  0.0s|     1 |     0 |     7 |     - |   941k |   0 |  15 |   2 |  14 |   0 |  0 |   0 |   0 |-2.919

wn
  0.0s|     1 |     0 |    57 |     - |  1027k |   0 |  15 |   2 |  64 |  50 | 10 |   0 |   0 |-8.240027e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    57 |     - |  1027k |   0 |  15 |   2 |  64 |  50 | 10 |   0 |   0 |-8.240027e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    62 |     - |  1041k |   0 |  15 |   2 |  57 |  55 | 11 |   0 |   0 |-8.240004e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    62 |     - |  1041k |   0 |  15 |   2 |  57 |  55 | 11 |   0 |   0 |-8.240004e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    67 |     - |  1041k |   0 |  15 |   2 |  62 |  60 | 12 |   0 |   0 |-8.240001e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    67 |     - |  1041k |   0 |  15 |   2 |  62 |  60 | 12 |   0 |   0 |-8.240001e+02 |-8.240000e+02 |   0.00%| unknown
  0.0s|     1 |     0 |    72 |     - |  1041k |   0 |  15 |   2 |  67 |  65 | 13 |   0 |   0 |-8.240000e+02 |-8.240000e+02 |   0.00%| unkn

3 |-3.219092e+02 | 370.86%| unknown
  0.0s|     1 |     0 |     7 |     - |  1112k |   0 |  15 |   3 |  15 |   0 |  0 |   0 |   0 |-1.515750e+03 |-3.219092e+02 | 370.86%| unknown
  0.0s|     1 |     0 |    12 |     - |  1112k |   0 |  15 |   3 |  20 |   5 |  1 |   0 |   0 |-7.563296e+02 |-3.219092e+02 | 134.95%| unknown
  0.0s|     1 |     0 |    12 |     - |  1112k |   0 |  15 |   3 |  20 |   5 |  1 |   0 |   0 |-7.563296e+02 |-3.219092e+02 | 134.95%| unknown
  0.0s|     1 |     0 |    17 |     - |  1112k |   0 |  15 |   3 |  25 |  10 |  2 |   0 |   0 |-3.943750e+02 |-3.219092e+02 |  22.51%| unknown
  0.0s|     1 |     0 |    17 |     - |  1112k |   0 |  15 |   3 |  25 |  10 |  2 |   0 |   0 |-3.943750e+02 |-3.219092e+02 |  22.51%| unknown
  0.0s|     1 |     0 |    21 |     - |  1112k |   0 |  15 |   3 |  30 |  15 |  3 |   0 |   0 |-3.494063e+02 |-3.219092e+02 |   8.54%| unknown
  0.0s|     1 |     0 |    21 |     - |  1112k |   0 |  15 |   3 |  30 |  15 |  3 |   0 |   0 |-3.494063e+

 |  73 |  71 | 15 |   0 |   0 |-4.637048e-01 |-4.637036e-01 |   0.00%| unknown
  0.0s|     1 |     0 |    71 |     - |  1121k |   0 |  15 |   4 |  77 |  75 | 16 |   0 |   0 |-4.637039e-01 |-4.637036e-01 |   0.00%| unknown
  0.0s|     1 |     0 |    71 |     - |  1121k |   0 |  15 |   4 |  77 |  75 | 17 |   0 |   0 |-4.637039e-01 |-4.637036e-01 |   0.00%| unknown
L 0.0s|     1 |     0 |    71 |     - |multista|   0 |  15 |   4 |  77 |  75 | 18 |   0 |   0 |-4.637039e-01 |-4.637036e-01 |   0.00%| unknown
  0.0s|     1 |     0 |    71 |     - |  1121k |   0 |  15 |   4 |  77 |  75 | 18 |   0 |   0 |-4.637039e-01 |-4.637036e-01 |   0.00%| unknown
* 0.0s|     1 |     0 |    71 |     - |    LP  |   0 |  15 |   4 |  77 |  75 | 19 |   0 |   0 |-4.637039e-01 |-4.637039e-01 |   0.00%| unknown

SCIP Status        : problem is solved [optimal solution found]
Solving Time (sec) : 0.04
Solving Nodes      : 1
Primal Bound       : -4.63703898685306e-01 (4 solutions)
Dual Bound         : -4.63703898685

s, 0 added constraints, 0 tightened bounds, 0 added holes, 0 changed sides, 0 changed coefficients
 0 implications, 0 cliques, 0 implied integral variables (0 bin, 0 int, 0 cont)
presolved problem has 8 variables (0 bin, 0 int, 8 cont) and 8 constraints
      7 constraints of type <linear>
      1 constraints of type <nonlinear>
Presolving Time: 0.00
transformed 2/2 original solutions to the transformed problem space

 time | node  | left  |LP iter|LP it/n|mem/heur|mdpt |vars |cons |rows |cuts |sepa|confs|strbr|  dualbound   | primalbound  |  gap   | compl. 
  0.0s|     1 |     0 |     9 |     - |  1133k |   0 |  15 |   8 |  20 |   0 |  0 |   0 |   0 |-5.083331e+01 | 1.823465e+03 |    Inf | unknown
L 0.0s|     1 |     0 |     9 |     - |  subnlp|   0 |  15 |   8 |  20 |   0 |  0 |   0 |   0 |-5.083331e+01 | 3.389650e+00 |    Inf | unknown
  0.0s|     1 |     0 |     9 |     - |  1133k |   0 |  15 |   8 |  20 |   0 |  0 |   0 |   0 |-5.083331e+01 | 3.389650e+00 |    Inf | unknown
  0.0s

In [13]:
optimal_weights

('NodeSelection', 'weight') = -7.118081452353971
('NodeSelection', 'constant') = 5.3154207936935505
('EdgeSelection', 'weight') = -9.95664493764966
('EdgeSelection', 'constant') = 3.5425539453511896
('Appear', 'weight') = 0.0
('Appear', 'constant') = 8.436845000403537e-09

In [14]:
solver = motile.Solver(graph)

solver.add_constraint(MaxParents(1))
solver.add_constraint(MaxChildren(1))

solver.add_cost(NodeSelection(weight=1, attribute="score"))
solver.add_cost(EdgeSelection(weight=-2, attribute="score", constant=1))
solver.add_cost(Appear(constant=1))

solver.weights.from_ndarray(optimal_weights.to_ndarray())

solver.solve()

Solution(variable_values=[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], objective_value=-7.436625719070435, status=<SolverStatus.OPTIMAL: 'optimal'>, time=0.0, native_status='optimal')

In [15]:
draw_solution(graph, solver, label_attribute="score")